In [ ]:
### Preliminaries
import meep as mp
from meep import mpb
import numpy as np
from photonic_crystal_waveguide import PhotonicCrystalWaveguideBands as PC
from ipywidgets import interact, IntSlider
import warnings
warnings.filterwarnings('ignore')
from mayavi import mlab
mlab.init_notebook()

In [ ]:
# Change this path to point to your simulated data (or where you want to save the data)
path = '/mnt/12B92FAC0F9C831C/Simulations_and_Scripts/mpb/data'
path = path
pc = PC()
pc.setPath(path)


## To run a single simulation from here uncomment the following code:
#epsilon = 5.847                # Material dielectric constant (n**2 = 2.148**2 = 5.847 for diamond)
#default_epsilon = 1            # Background material dielectric constant
#cross_section = 'triangular'   # Beam cross-section
#hx = 0.313                     # Hole y radius
#hy = 0.333                     # Hole x radius
#width = 1.956                  # Beam width
#height = 0.4                   # Beam height, not used for triangular cross-section simulations
#angle = 0.873                  # Beam half-angle in radians, not used for rectangular cross-section simulations
#dboundary = 1                  # Distance to simulation periodic boundary

# Simulation Parameters
#resolution = 10                # Unit cell discretization points (just of dielectric material), RMS
#                               # error is ~1e-4 at resolution = 10 and decreases at ~10 dB/ 7
#                               # points, marginal gains by increasing this
#num_bands = 4                  # Number of bands to calculate
#k_start = 0.3                  # Starting k point (unitless)
#k_stop = 0.5                   # Ending k points (max 0.5, X point)
#num_kpoints = 15               # Number of k points

#pc.buildGeometry(epsilon, hx, hy, width, height=height, default_epsilon=default_epsilon, cross_section=cross_section, angle=angle)
#pc.setupSimulation(resolution, num_bands, k_start, k_stop, num_interpolation_points, default_epsilon)

#pc.run()
#pc.saveData()                  #Saves data to path

pc.loadData()                  #disable if running single simulation above

In [ ]:
### Plotting the Dielectric Constant in 3D:
##  Plots the dielectric constant in 3D space on a discretized grid, by deault the data plotted modulates the
##  pixel cube's size and color, here I have overriden the color and only modulate the pizel size. Plotted
##  data offset such that the air background is zero and normalized to 1 (contacting pixel cubes)

# Color definition in (r,g,b)
white = (1,1,1)

# Paramters
periods = 1
color = white
mask_points = 1
mode = 'cube'
opacity = 0.35
scale_factor = 1

#md = mpb.MPBData(rectify=True, periods=periods, resolution=resolution)
#eps = pc.te_ms.get_epsilon()
#converted_eps = md.convert(eps)
#epsilon = converted_eps
epsilon = pc.epsilon_geometry

# 3D plot
ny, nx, nz = epsilon.shape
x = np.arange(0, nx)
y = np.arange(0, ny)
z = np.arange(0, nz)
X,Y,Z = np.meshgrid(x,y,z)

fig = mlab.figure()
mlab.clf(fig)
mlab.points3d(X,Y,Z,(epsilon-1)/(pc.geometry_parameters['epsilon'] - 1), color=color, mask_points=mask_points,
              mode=mode, opacity=opacity, scale_factor = scale_factor)
mlab.orientation_axes()

In [ ]:
a = 436                         # Unit cell length in nm
polarization = 'all'            # Can be 'TE', 'TM', or 'all'

list_units = None              # Units can be None, 'THz', or nm
list_gaps = 'one'               # 'all' or 'one'

plot_units = 'THz'              # Units can be None, 'THz'
plot_gaps = 'one'               # 'all' or 'one'

pc.calcBandGaps(condition='X')  # Condition 'X' calculates bands at the X point, 'all' considers all k-points
                                # below the light line
pc.listGaps(polarization=polarization, units = list_units, gaps=list_gaps, a=a)      
print("Unit cell length for a TE midgap of 737 nm:",
      737 * pc.te_quasi_gaps_df['Midgap ($\omega$a/2$\pi$c)'][0], ' nm')
pc.plotBands(polarization=polarization, gaps=plot_gaps, a=a, units = plot_units, invert=False, save=False)

In [ ]:
### Plotting mode profiles at the X-point

# Color definition in (r,g,b)
white = (1,1,1)

## Field Plot Parameters:
## field: Allowed vector fields are 'H', 'D', 'E', and 'S'
#         Scalar fields specified by appending the component (x, y, z, m - magnitide),
#         e.g. 'Hx', 'Hy', 'Hz', 'Hm'
#         Energy desnity of the component can be found by then appending E e.g. 'HxE', 'HyE', 'HzE', 'HmE'
## polarization: Allowed polarizations are 'TE' and 'TM' 
## band: Allow bands range from 1 to num_bands 
## part: 'real', 'imaginary', 'magnitude', 'argument', or 'max'
#        Unfortunatuely the field component phase is arbitrary (though not random), for the sake of plotting,
#        for a given band these simulations fix either E or H to be completely real meaninging the other will
#        be purely imaginary. 'max' chooses whichever of the real of imaginary part is larger
## components: 'x', 'y', 'z', or None, for vector field slice plots, the field to plot must be specified
## mode: I prefer '2darrow' for 3D quiver plots and 'cube' for scalar field plots for all options refer to
#       the mayavi documentation: https://docs.enthought.com/mayavi/mayavi/auto/mlab_helper_functions.html
## scale_factor: A multiplier for the size of the pixels/arrows, I usually leave it at one for scalar fields
#                but vector fields often need a bit of amplification for good visibility
## opacity: Number between 0 and 1 specifying transparency, scalar fields need to be quite transparent to still
#           see the dielectric. Vector fields can be more opaque, just need to be made sparser
## mask_points: An integer specifying how often to plot a data point, everything in between will be ignored,
#               useful for making data sparser in e.g. 3D quiver plots

field = 'Hz'
polarization = 'TE'
band = 1
part = 'max'
bloch_phase = True
if field == 'D' or field == 'H' or field == 'E' or field == 'S':
    # Field is a vector
    pc.xz_component = 'z'
    pc.yz_component = 'z'
    pc.xy_component = 'z'
    field_mode = '2darrow'
    field_scale_factor = 5
    field_opacity = 0.5
    field_mask_points = 15
else:
    # Field is a scalar
    pc.xz_component = None
    pc.yz_component = None
    pc.xy_component = None
    field_mode = 'cube'
    field_scale_factor = 1
    field_opacity = 0.05
    field_mask_points = 1

# Beam Plot Paramters
beam_color = white
beam_mask_points = 1
beam_mode = 'cube'
beam_opacity = 0.5
beam_scale_factor = 1

# Get field Data
pc.getField(field, polarization, band, part=part)

# 2D Slice Plots
print('\n2D ' + field + ' Field Slices')
interact(pc.plotSlices,
         x=IntSlider(value=0, min=-pc.nx//2, max=pc.nx//2 - 1),
         y=IntSlider(value=0, min=-pc.ny//2, max=pc.ny//2 - 1),
         z=IntSlider(value=0, min=-pc.nz//2, max=pc.nz//2 - 1))

# 3D Field Plot
x = np.arange(0, pc.nx)
y = np.arange(0, pc.ny)
z = np.arange(0, pc.nz)
X,Y,Z = np.meshgrid(x,y,z)
print('\n' + field + ' field')
fig = mlab.figure()
mlab.clf(fig)
mlab.points3d(Y, X, Z, (pc.epsilon_geometry - 1) / (pc.geometry_parameters['epsilon'] - 1),
              color=beam_color, mask_points=beam_mask_points, mode=beam_mode, opacity=beam_opacity,
              scale_factor = beam_scale_factor)
if pc.field_type == 'scalar':
    f = mlab.points3d(Y, X, Z, pc.field_data / np.max(pc.field_data),
                    mask_points=field_mask_points, mode=field_mode, opacity=field_opacity,
                    scale_factor = field_scale_factor, colormap='plasma')
elif pc.field_type == 'vector':
    f = mlab.quiver3d(Y, X, Z, pc.field_data[:, :, :, 1], pc.field_data[:, :, :, 0], pc.field_data[: , :, :, 2],
                    mask_points=field_mask_points, mode=field_mode, opacity=field_opacity,
                    scale_factor = field_scale_factor, colormap='plasma')
mlab.orientation_axes()
mlab.colorbar(f)

In [ ]:
# Display the fraction of energy in each field component
print('Distribution of energy in field components')
display(pc.power_fractions_df)

In [ ]:
# Display group veloities for each band and polarization
print('TE Group Velocity Components')
display(pc.te_vg_df)
print('TM Group Velocity Components')
display(pc.tm_vg_df)

In [ ]:
# Analysis of convergence parameter sweeps: dboundary

#import os
#from analyze_sweep import Sweep
#os.chdir('/home/graham/Documents/mpb/')
#sweep_directory = '/home/graham/Documents/mpb/data/dboundary_sweep/'
#sw = Sweep(sweep_directory)
#sw.sweep_parameter = 'dboundary'
#sw.start = 1
#sw.stop = 28
#interact(sw.plotConvergence, k_ind = IntSlider(value = sw.num_kpoints, min=0, max=sw.num_kpoints-1), legend=True, log=False);

In [ ]:
# Analysis of convergence parameter sweeps: resolution

#from analyze_sweep import Sweep
#os.chdir('/home/graham/Documents/mpb/')
#sweep_directory = '/home/graham/Documents/mpb/data/resolution_sweep/'
#sw = Sweep(sweep_directory)
#sw.sweep_parameter = 'resolution'
#sw.start = 0
#sw.stop = 23
#interact(sw.plotConvergence, k_ind = IntSlider(value = sw.num_kpoints, min=0, max=sw.num_kpoints-1), legend=True, log=False);